# Yotuubef Colab: Web Studio GUI

This notebook launches the complete interactive **Yotuubef Web Studio** graphical user interface.

Before running:
1. Set **Runtime -> Change runtime type -> GPU** (recommended for video rendering & voiceover TTS).
2. Add Colab Secrets (left sidebar key icon 🔑): `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, `NVIDIA_NIM_API_KEY`, `YOUTUBE_TOKEN_JSON` (optional).
3. Put one or more `.mp4` background videos in your Drive folder (`/content/drive/MyDrive/yotuubef/persistent/backgrounds`).

Workflow:
- **Cell 1-3**: Setup environment, mount Google Drive, load secrets.
- **Cell 4**: Launch Web Studio GUI -> Click the link button to open the Web App!

In [1]:
# Cell 1: Environment Setup & Dependencies
import os
import shlex
import subprocess
from pathlib import Path

def run(cmd: str, allow_fail: bool = False) -> None:
    print(f"$ {cmd}")
    completed = subprocess.run(cmd, shell=True, text=True)
    if completed.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({completed.returncode}): {cmd}")

# 1. System dependencies
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg sox git gcc g++ nodejs npm")
run("npm install -g n && n 22", allow_fail=True)

# 2. Clone repo
REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_DIR = Path("/content/yotuubef")

if not REPO_DIR.exists():
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")

os.chdir(REPO_DIR)
run("git pull --ff-only || true", allow_fail=True)

# 3. Python dependencies
run("python3 -m pip install -q --upgrade pip setuptools wheel")
run("python3 -m pip install -q -r requirements.txt")
run("python3 -m pip install -q nest_asyncio ipywidgets")

# 4. Qwen3-TTS (required for voiceover generation)
qwen_cmd = "python3 -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git"
print(f"$ {qwen_cmd}")
qwen_res = subprocess.run(qwen_cmd, shell=True, text=True)
if qwen_res.returncode != 0:
    print("Warning: Qwen3-TTS install failed. Pipeline can still run with fallback TTS.")

# 5. Build Web Studio Frontend
gui_dir = REPO_DIR / "gui"
if (gui_dir / "package.json").exists():
    print("🔨 Building Web Studio frontend...")
    subprocess.run(["npm", "install"], cwd=gui_dir, check=True)
    subprocess.run(["npm", "run", "build"], cwd=gui_dir, check=True)

print("✅ Setup complete.")

$ apt-get -qq update
$ apt-get -qq install -y ffmpeg sox git gcc g++ nodejs npm
$ npm install -g n && n 22
$ git clone https://github.com/beenycool/yotuubef.git /content/yotuubef
$ git pull --ff-only || true
$ python3 -m pip install -q --upgrade pip setuptools wheel
$ python3 -m pip install -q -r requirements.txt
$ python3 -m pip install -q nest_asyncio ipywidgets
$ python3 -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git
🔨 Building Web Studio frontend...
✅ Setup complete.


In [2]:
# Cell 2: Mount Drive & Configure Persistence
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)

DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
PERSIST_ROOT = DRIVE_ROOT / "persistent"
RUNS_ROOT = DRIVE_ROOT / "runs"

# Define persistent directories
important_dirs = {
    "backgrounds": PERSIST_ROOT / "backgrounds",
    "findings": PERSIST_ROOT / "findings",
    "processed": PERSIST_ROOT / "processed",
    "results": PERSIST_ROOT / "results",
    "logs": PERSIST_ROOT / "logs",
    "db": PERSIST_ROOT / "databases",
    "tokens": PERSIST_ROOT / "tokens",
}

for p in [DRIVE_ROOT, PERSIST_ROOT, RUNS_ROOT, *important_dirs.values()]:
    p.mkdir(parents=True, exist_ok=True)

def relink_dir(local_path: Path, drive_path: Path) -> None:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve(strict=False) == drive_path.resolve(strict=False):
            return
        local_path.unlink()
    elif local_path.exists():
        if not local_path.is_dir():
            raise RuntimeError(f"Cannot relink non-directory: {local_path}")
        for item in local_path.iterdir():
            target_item = drive_path / item.name
            if target_item.exists():
                raise RuntimeError(f"Target already exists: {target_item}")
        shutil.rmtree(local_path)
    local_path.symlink_to(drive_path, target_is_directory=True)

link_map = {
    REPO_DIR / "findings": important_dirs["findings"],
    REPO_DIR / "processed": important_dirs["processed"],
    REPO_DIR / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "results": important_dirs["results"],
    REPO_DIR / "data" / "logs": important_dirs["logs"],
    REPO_DIR / "data" / "databases": important_dirs["db"],
}

for local_dir, drive_dir in link_map.items():
    relink_dir(local_dir, drive_dir)

print("✅ Drive mounted and persistence linked.")
print(f"📁 Background folder: {important_dirs['backgrounds']}")

Mounted at /content/drive
✅ Drive mounted and persistence linked.
📁 Background folder: /content/drive/MyDrive/yotuubef/persistent/backgrounds


In [3]:
# Cell 3: Secrets & Preflight Check
import json
import os
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
BACKGROUND_FOLDER_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/backgrounds")
TOKEN_FILE_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/tokens/youtube_token.json")

def get_secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        value = userdata.get(name)
        return value if value is not None else default
    except Exception:
        return os.getenv(name, default)

# Map secrets to env vars
secret_map = {
    "REDDIT_CLIENT_ID": "REDDIT_CLIENT_ID",
    "REDDIT_CLIENT_SECRET": "REDDIT_CLIENT_SECRET",
    "NVIDIA_NIM_API_KEY": "NVIDIA_NIM_API_KEY",
    "HACKCLUB_SEARCH_API_KEY": "HACKCLUB_SEARCH_API_KEY",
    "YOUTUBE_TOKEN_JSON": "YOUTUBE_TOKEN_JSON",
}

for secret_name, env_name in secret_map.items():
    val = get_secret(secret_name, "")
    if val:
        os.environ[env_name] = val

os.environ["BACKGROUND_FOLDER"] = str(BACKGROUND_FOLDER_PATH)
os.environ["YOUTUBE_TOKEN_FILE"] = str(TOKEN_FILE_PATH)

youtube_token_json = get_secret("YOUTUBE_TOKEN_JSON", "")
if youtube_token_json:
    try:
        parsed = json.loads(youtube_token_json)
        TOKEN_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE_PATH.write_text(json.dumps(parsed), encoding="utf-8")
        print(f"Wrote YouTube token file to: {TOKEN_FILE_PATH}")
    except Exception as exc:
        print(f"Warning: Could not parse/write YOUTUBE_TOKEN_JSON ({exc})")

# --- PREFLIGHT CHECK ---
print("🔍 Running Preflight Checks...")
errors = []

if not os.environ.get("REDDIT_CLIENT_ID"):
    errors.append("⚠️ REDDIT_CLIENT_ID missing in Colab Secrets (can set in Web GUI Settings)")
if not os.environ.get("REDDIT_CLIENT_SECRET"):
    errors.append("⚠️ REDDIT_CLIENT_SECRET missing in Colab Secrets (can set in Web GUI Settings)")
if not os.environ.get("NVIDIA_NIM_API_KEY"):
    errors.append("⚠️ NVIDIA_NIM_API_KEY missing in Colab Secrets (can set in Web GUI Settings)")

bg_videos = list(BACKGROUND_FOLDER_PATH.glob("*.mp4"))
if not bg_videos:
    print(f"⚠️ Warning: No .mp4 background videos found in {BACKGROUND_FOLDER_PATH}")
else:
    print(f"✅ Found {len(bg_videos)} background videos.")

print("✅ Preflight check complete.")

🔍 Running Preflight Checks...
✅ Found 1 background videos.
✅ Preflight check complete.


In [8]:
    import os
    import sys
    import importlib
    from pathlib import Path

    REPO_DIR = Path("/content/yotuubef")
    os.chdir(REPO_DIR)
    !git pull origin main

    import start_studio
    importlib.reload(start_studio)
    from start_studio import run_colab_studio

    print("🚀 Launching Yotuubef Studio Web App...")
    run_colab_studio(port=8420)

From https://github.com/beenycool/yotuubef
 * branch            main       -> FETCH_HEAD
Already up to date.
🚀 Launching Yotuubef Studio Web App...
🚀 Launching FastAPI backend + Web Studio server on port 8420...


✨ Studio URL: https://8420-m-s-kkb-use4a0-2bu9zdu67se61-a.us-east4-0.prod.colab.dev


<Popen: returncode: None args: ['/usr/bin/python3', '-m', 'uvicorn', 'src.ap...>